# XVI detector image viewer

Load the six independent Bristol XVI energy-deposition images written by the Geant4 simulation. Each image is a `1024 x 1024` float32 array in MeV with matching JSON metadata.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})

OUTPUT_DIR = Path("PhotonDetector/build")
STEM = "PSfile"
PANEL_ORDER = ["pos_x", "neg_x", "pos_y", "neg_y", "pos_z", "neg_z"]

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"{OUTPUT_DIR.resolve()} does not exist. Start Jupyter from the xvi-detector worktree root, "
        "or change OUTPUT_DIR above."
    )

## Load and validate the panel files

In [ ]:
def load_panel(raw_path: Path):
    metadata_path = raw_path.with_suffix(".json")
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing metadata file: {metadata_path}")

    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    rows = int(metadata["rows"])
    columns = int(metadata["columns"])
    flat = np.fromfile(raw_path, dtype=np.float32)
    expected = rows * columns
    if flat.size != expected:
        raise ValueError(f"{raw_path} contains {flat.size} values; expected {expected}")

    return flat.reshape(rows, columns), metadata


def discover_panels(output_dir: Path, stem: str):
    panels = {}
    pattern = f"{stem}_xvi_*_edep.raw"
    for raw_path in sorted(output_dir.glob(pattern)):
        image, metadata = load_panel(raw_path)
        panels[metadata["panel_name"]] = {
            "image": image,
            "metadata": metadata,
            "raw_path": raw_path,
        }
    if not panels:
        raise FileNotFoundError(f"No files matching {output_dir / pattern}")
    return panels


panels = discover_panels(OUTPUT_DIR, STEM)
print(f"Loaded {len(panels)} panels from {OUTPUT_DIR.resolve()}")
print("Panels:", ", ".join(name for name in PANEL_ORDER if name in panels))

## Run summary

`hit_steps` is the number of energy-depositing Geant4 steps, not the number of incident photons.

In [ ]:
header = f"{'panel':<8} {'nonzero pixels':>15} {'hit steps':>12} {'image sum (MeV)':>18} {'maximum (MeV)':>16}"
print(header)
print("-" * len(header))
for name in PANEL_ORDER:
    if name not in panels:
        continue
    image = panels[name]["image"]
    metadata = panels[name]["metadata"]
    print(
        f"{name:<8} {np.count_nonzero(image):>15,d} {metadata['hit_steps']:>12,d} "
        f"{image.sum(dtype=np.float64):>18.9g} {image.max():>16.9g}"
    )

## Six panels on one physical and colour scale

This view is appropriate for comparing absolute signal between panels. Weak panels can look blank because all images share the same scale.

In [ ]:
available = [panels[name] for name in PANEL_ORDER if name in panels]
global_max = max(item["image"].max() for item in available)
global_max = max(float(global_max), np.finfo(np.float32).eps)

fig, axes = plt.subplots(3, 2, figsize=(11, 14), constrained_layout=True)
last_image = None
for ax, name in zip(axes.flat, PANEL_ORDER):
    if name not in panels:
        ax.set_visible(False)
        continue
    item = panels[name]
    image = item["image"]
    metadata = item["metadata"]
    half_size = 0.5 * metadata["panel_size_mm"]
    last_image = ax.imshow(
        image,
        origin="lower",
        extent=(-half_size, half_size, -half_size, half_size),
        cmap="magma",
        vmin=0.0,
        vmax=global_max,
        interpolation="nearest",
    )
    ax.set_title(f"{name}: {image.sum(dtype=np.float64):.4g} MeV")
    ax.set_xlabel("Local x (mm)")
    ax.set_ylabel("Local y (mm)")
    ax.set_aspect("equal")

fig.colorbar(last_image, ax=axes.ravel().tolist(), shrink=0.75, label="Deposited energy (MeV / pixel)")
fig.suptitle(f"{STEM}: independent XVI panels, shared linear scale")
plt.show()

## Per-panel logarithmic contrast

Each non-empty panel is scaled independently. This reveals sparse or weak deposits but must not be used to compare absolute intensity by colour.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 14), constrained_layout=True)
for ax, name in zip(axes.flat, PANEL_ORDER):
    if name not in panels:
        ax.set_visible(False)
        continue
    item = panels[name]
    image = item["image"]
    metadata = item["metadata"]
    positive = image[image > 0]
    half_size = 0.5 * metadata["panel_size_mm"]
    ax.set_title(name)
    ax.set_xlabel("Local x (mm)")
    ax.set_ylabel("Local y (mm)")
    ax.set_aspect("equal")
    if positive.size == 0:
        ax.text(0.5, 0.5, "No deposited energy", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlim(-half_size, half_size)
        ax.set_ylim(-half_size, half_size)
        continue
    vmin = float(positive.min())
    vmax = max(float(positive.max()), vmin * (1.0 + 1e-6))
    plotted = ax.imshow(
        np.ma.masked_less_equal(image, 0),
        origin="lower",
        extent=(-half_size, half_size, -half_size, half_size),
        cmap="viridis",
        norm=LogNorm(vmin=vmin, vmax=vmax),
        interpolation="nearest",
    )
    fig.colorbar(plotted, ax=ax, shrink=0.75, label="MeV / pixel")

fig.suptitle(f"{STEM}: independent per-panel log scales")
plt.show()

## Selected-panel zoom and centroid

Change `SELECTED_PANEL` to inspect another readout. The view automatically crops around nonzero pixels while preserving physical coordinates.

In [ ]:
SELECTED_PANEL = "pos_y"
ZOOM_MARGIN_PIXELS = 20

item = panels[SELECTED_PANEL]
image = item["image"]
metadata = item["metadata"]
pitch = float(metadata["pixel_pitch_mm"])
half_size = 0.5 * float(metadata["panel_size_mm"])
nonzero = np.argwhere(image > 0)

if nonzero.size:
    y0, x0 = np.maximum(nonzero.min(axis=0) - ZOOM_MARGIN_PIXELS, 0)
    y1, x1 = np.minimum(nonzero.max(axis=0) + ZOOM_MARGIN_PIXELS + 1, image.shape)
    crop = image[y0:y1, x0:x1]
    extent = (x0 * pitch - half_size, x1 * pitch - half_size, y0 * pitch - half_size, y1 * pitch - half_size)
    weights = image[nonzero[:, 0], nonzero[:, 1]].astype(np.float64)
    centroid_x = np.average((nonzero[:, 1] + 0.5) * pitch - half_size, weights=weights)
    centroid_y = np.average((nonzero[:, 0] + 0.5) * pitch - half_size, weights=weights)

    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
    plotted = ax.imshow(crop, origin="lower", extent=extent, cmap="magma", interpolation="nearest")
    ax.scatter([centroid_x], [centroid_y], marker="+", s=120, linewidths=1.5, color="cyan", label="Energy-weighted centroid")
    ax.set_title(f"{SELECTED_PANEL}: signal-region zoom")
    ax.set_xlabel("Local x (mm)")
    ax.set_ylabel("Local y (mm)")
    ax.legend()
    fig.colorbar(plotted, ax=ax, label="Deposited energy (MeV / pixel)")
    plt.show()
    print(f"Centroid: x={centroid_x:.3f} mm, y={centroid_y:.3f} mm")
else:
    print(f"{SELECTED_PANEL} contains no deposited energy.")

## Integrated local-axis profiles

In [ ]:
centres_mm = (np.arange(image.shape[1]) + 0.5) * pitch - half_size
x_profile = image.sum(axis=0, dtype=np.float64)
y_profile = image.sum(axis=1, dtype=np.float64)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(centres_mm, x_profile, color="tab:blue")
axes[0].set(title=f"{SELECTED_PANEL}: x profile", xlabel="Local x (mm)", ylabel="Summed energy (MeV)")
axes[1].plot(centres_mm, y_profile, color="tab:orange")
axes[1].set(title=f"{SELECTED_PANEL}: y profile", xlabel="Local y (mm)", ylabel="Summed energy (MeV)")
plt.show()